# Week 12, Day 6: Exam debrief

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w012d6.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


In [ ]:
def normalise(number):
    digits = number.replace(" ", "").replace("-", "")
    for prefix in ("+234", "234", "0"):
        if digits.startswith(prefix):
            rest = digits[len(prefix):]
            if len(rest) == 10 and rest.isdigit() and rest[0] != "0":
                return "+234" + rest
            break
    raise ValueError(f"invalid number: {number!r}")

for t in ["0803 123 4567", "+234 803-123-4567", "0803 123 456"]:
    try:
        print(normalise(t))
    except ValueError as e:
        print(e)

In [ ]:
def runs(text):
    result = []
    for ch in text:
        if result and result[-1][0] == ch:
            result[-1] = (ch, result[-1][1] + 1)
        else:
            result.append((ch, 1))
    return result

def expand(pairs):
    return "".join(ch * n for ch, n in pairs)

print(runs("aaabccdd"))
print(expand(runs("mississippi")))

In [ ]:
def merge(a, b):
    i = j = 0
    result = []
    while i < len(a) and j < len(b):
        if a[i] <= b[j]:
            result.append(a[i])
            i += 1
        else:
            result.append(b[j])
            j += 1
    return result + a[i:] + b[j:]

print(merge([1, 4, 9], [2, 3, 10, 11]))

In [ ]:
def spiral(grid):
    rows = [list(r) for r in grid]
    result = []
    while rows:
        result += rows.pop(0)
        rows = [list(col) for col in zip(*rows)][::-1]
    return result

print(spiral([[1, 2, 3], [4, 5, 6], [7, 8, 9]]))

These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('register.txt', 'w', encoding='utf-8') as f:
    f.write('2026-10-05 Ada, Bo, Cy\n2026-10-06 Ada,Cy\n\n2026-10-07 Bo, Cy\n2026-10-08\n2026-10-09 Ada, Bo, Cy, Di\n')

In [ ]:
def attendance(path):
    days = 0
    present = {}
    with open(path) as f:
        for line in f:
            if not line.strip():
                continue
            days += 1
            date, _, names = line.strip().partition(" ")
            for name in names.split(","):
                name = name.strip()
                if name:
                    present[name] = present.get(name, 0) + 1
    return {name: round(100 * n / days, 1) for name, n in present.items()}

print(attendance("register.txt"))

In [ ]:
PAIRS = {")": "(", "]": "[", "}": "{"}
CLOSER = {"(": ")", "[": "]", "{": "}"}

def check_brackets(text):
    stack = []
    for i, ch in enumerate(text):
        if ch in CLOSER:
            stack.append((ch, i))
        elif ch in PAIRS:
            if not stack:
                return f"unexpected {ch} at {i}"
            opener, _ = stack.pop()
            if opener != PAIRS[ch]:
                return f"expected {CLOSER[opener]} but found {ch} at {i}"
    if stack:
        opener, i = stack[-1]
        return f"{opener} at {i} never closed"
    return "OK"

for t in ["f(x[0], {1: (2)})", "(a[b(c)) ", "a (b [c] d"]:
    print(repr(t), check_brackets(t))

These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('parking.csv', 'w', encoding='utf-8') as f:
    f.write('plate,time,event\nLAG 123 AA,07:55,in\nabj 45 kd,08:10,in\nKJA 9 XY,08:12,in\nLAG123AA,08:20,out\nABJ 45 KD,09:41,out\nEKY 7 BC,09:50,out\nKJA 9 XY,10:05,in\nOYO 1 ZZ,25:00,in\nKJA 9 XY,16:30,out\nLAG 123 AA,17:00,in\nLAG 123 AA,17:01,fly\n')

In [ ]:
import csv

def minutes(time):
    parts = time.split(":")
    if len(parts) != 2 or [len(p) for p in parts] != [2, 2] or not all(p.isdigit() for p in parts):
        raise ValueError(f"bad time {time!r}")
    h, m = int(parts[0]), int(parts[1])
    if h > 23 or m > 59:
        raise ValueError(f"bad time {time!r}")
    return h * 60 + m

def fee(stay):
    if stay <= 30:
        return 0
    hours = (stay + 59) // 60
    return min(200 * hours, 1500)

def process(path):
    inside, charges, problems = {}, [], []
    with open(path, newline="") as f:
        for row_no, row in enumerate(csv.DictReader(f), start=2):
            plate = row["plate"].replace(" ", "").upper()
            event = row["event"].strip()
            try:
                t = minutes(row["time"].strip())
            except ValueError:
                problems.append(f"row {row_no}: bad time {row['time'].strip()!r}")
                continue
            if event == "in":
                if plate in inside:
                    problems.append(f"row {row_no}: {plate} entered twice")
                else:
                    inside[plate] = t
            elif event == "out":
                if plate not in inside:
                    problems.append(f"row {row_no}: {plate} left without entering")
                else:
                    stay = t - inside.pop(plate)
                    charges.append((plate, stay, fee(stay)))
            else:
                problems.append(f"row {row_no}: unknown event {event!r}")
    for plate in sorted(inside):
        problems.append(f"{plate} never left")
    return charges, problems

charges, problems = process("parking.csv")
for plate, stay, cost in charges:
    print(f"{plate:<10}{stay:>6} min{cost:>8}")
print(f"Total: {sum(c for _, _, c in charges)} naira")
if problems:
    print("Problems:")
    for p in problems:
        print(" ", p)
else:
    print("Problems: none")

### Exercise: Second chance: email addresses

(Like B1.) Write `clean_email(text)` that strips spaces and lower-cases an email address, then checks it: exactly one `@`, at least one character before it, and after it a domain with at least one `.` that isn't the first or last character of the domain. Return the clean address, or raise a `ValueError` with the message `invalid email: ` and the original text in quotes.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def bad(t):\n    try:\n        clean_email(t)\n    except ValueError as e:\n        assert str(e) == f\"invalid email: {t!r}\"\n        return\n    raise AssertionError(f\"{t!r} should be rejected\")\ndef test_values():\n    \"several addresses\"\n    assert clean_email(\"  Ada@Example.COM \") == \"ada@example.com\"\n    assert clean_email(\"a.b@x.co.uk\") == \"a.b@x.co.uk\"\n    for t in [\"ada.example.com\", \"@x.com\", \"a@b@c.com\", \"a@com\", \"a@.com\", \"a@com.\", \"\"]:\n        bad(t)")

### Exercise: Second chance: compressed text

(Like B2.) Write `compress(text)` that writes each run as the character followed by its count, but leaves out counts of 1: `compress("aaabccdd")` is `"a3bc2d2"`. Assume the text has no digits.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert compress(\"aaabccdd\") == \"a3bc2d2\"\n    assert compress(\"\") == \"\"\n    assert compress(\"abc\") == \"abc\"\n    assert compress(\"zzzzzzzzzzzz\") == \"z12\"")

### Exercise: Second chance: merge many

(Like B4.) Using the `merge` function given, write `merge_all(lists)` that merges any number of sorted lists into one sorted list, again without `sort` or `sorted`.

In [ ]:
def merge(a, b):
    i = j = 0
    result = []
    while i < len(a) and j < len(b):
        if a[i] <= b[j]:
            result.append(a[i])
            i += 1
        else:
            result.append(b[j])
            j += 1
    return result + a[i:] + b[j:]

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert merge_all([[1, 5], [2, 3], [0, 9]]) == [0, 1, 2, 3, 5, 9]\n    assert merge_all([]) == []\n    assert merge_all([[4]]) == [4]\n    assert \"sort\" not in source")

### Exercise: Second chance: diagonals

(Like B5.) Write `diagonals(grid)` that returns the anti-diagonals of a rectangular grid: lists of the items where row + column is 0, then 1, then 2, and so on, each from the top row down.

```
1 2 3
4 5 6    ->  [[1], [2, 4], [3, 5], [6]]
```

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert diagonals([[1, 2, 3], [4, 5, 6]]) == [[1], [2, 4], [3, 5], [6]]\n    assert diagonals([[1], [2], [3]]) == [[1], [2], [3]]\n    assert diagonals([]) == []")

### Exercise: Second chance: matching tags

(Like B8.) In a simplified HTML, tags look like `<b>` (opening) and `</b>` (closing), with only letters between the brackets. Write `check_tags(text)` that returns `"OK"` if every tag is closed in the right order, otherwise `unexpected </i>` (a closing tag that doesn't match the most recent open one, or with none open) or `<b> never closed` (the last tag still open at the end).

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert check_tags(\"<b>bold <i>both</i></b>\") == \"OK\"\n    assert check_tags(\"plain text\") == \"OK\"\n    assert check_tags(\"<b><i></b></i>\") == \"unexpected </b>\"\n    assert check_tags(\"</p>\") == \"unexpected </p>\"\n    assert check_tags(\"<p><b>hi</b>\") == \"<p> never closed\"")

### Exercise: Second chance: meeting rooms

(Like Part C.) Each booking is a tuple `(room, start, end)` with times as `HH:MM`. Using the `minutes` function given, write `clashes(bookings)` that returns a sorted list of `(room, i, j)` tuples for every pair of bookings `i < j` (their positions in the list) for the same room whose times overlap. Two bookings overlap if each starts before the other ends: back to back (`09:00 to 10:00` and `10:00 to 11:00`) is fine.

In [ ]:
import csv

def minutes(time):
    parts = time.split(":")
    if len(parts) != 2 or [len(p) for p in parts] != [2, 2] or not all(p.isdigit() for p in parts):
        raise ValueError(f"bad time {time!r}")
    h, m = int(parts[0]), int(parts[1])
    if h > 23 or m > 59:
        raise ValueError(f"bad time {time!r}")
    return h * 60 + m

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    b = [(\"A\", \"09:00\", \"10:00\"), (\"A\", \"10:00\", \"11:00\"), (\"B\", \"09:30\", \"10:30\"), (\"A\", \"09:30\", \"10:15\"), (\"B\", \"10:29\", \"12:00\")]\n    assert clashes(b) == [(\"A\", 0, 3), (\"A\", 1, 3), (\"B\", 2, 4)]\n    assert clashes([]) == []")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## Your score

Write down your total before reading on. Then, as you go through each problem below, note **why** you lost any marks. There are only a few kinds of reason, and knowing which kind you make most is more useful than the score itself:

1. **Didn't know it**: a gap in what you've learned. Revise the topic.
2. **Misread the question**: the code did something sensible but different. Slow down; underline every requirement.
3. **Edge case**: the main case worked, but empty input, a tie, or a boundary failed. Test small and extreme cases.
4. **Ran out of time**: the plan, not the knowledge, was the problem. Practise starting with the parts you're sure of.

| Score | What it means |
|---|---|
| 144 or more (80%) | Distinction. You're well prepared for Phase 2. |
| 108 to 143 (60%) | Pass. Revise the topics where you lost marks, then carry on. |
| under 108 | Not yet. Spend this week's rest days and a few more on the topics below, then take the exam again. Many people need two attempts; what matters is the second. |

## Part A

The quiz checks the ideas that cause most bugs. If you missed any, here's the idea behind each, and where it was taught:

| Questions | The idea | Revise |
|---|---|---|
| 1, 2, 3 | `//` and `%` round down; slices with steps; strings compare character by character | [Week 1](w001d4.html), [Week 1 Day 5](w001d5.html) |
| 4, 5 | tracing `while` loops and nested loops with `break` | [Week 3](w003d4.html), [Week 4](w004d3.html) |
| 6 | strings are sequences, so `join` works on them directly | [Week 5](w005d2.html) |
| 7, 8 | default values, `*args`, local names | [Week 6](w006d4.html), [Week 7](w007d2.html) |
| 9, 10, 11 | rebinding versus mutating; list methods | [Week 8](w008d5.html), [Week 8 Day 3](w008d3.html) |
| 12, 13 | tuples holding mutable values; comprehensions | [Week 9](w009d3.html), [Week 9 Day 5](w009d5.html) |
| 14, 15, 16 | `zip`, `setdefault`, and why 1, 1.0 and True are the same key | [Week 10](w010d2.html), [Week 10 Day 5](w010d5.html) |
| 17, 18, 19 | file modes, `finally`, raising instead of returning a special value | [Week 11](w011d2.html), [Week 11 Day 4](w011d4.html), [Week 11 Day 5](w011d5.html) |
| 20 | hashing | [Week 10 Day 1](w010d1.html) |

Question 16 surprises almost everyone. `1 == 1.0 == True` is `True` in Python, and equal values must hash the same, so a set treats them as one element. It's a curiosity, but it's a real one: a dictionary with keys `1` and `True` has one entry.

## Part B

Each model solution runs, so try it. The notes underneath each one explain the key decision and the most common mistakes.

### B1: Phone numbers

**The key idea:** clean first, then check the **structure** (prefix, then exactly 10 digits not starting with 0) rather than trying to fix every possible input. Checking prefixes from longest to shortest matters: `"+234"` must be tried before `"234"`.

**Common mistakes:** forgetting `isdigit` (so `0803 abc 4567` slips through after removing spaces), checking the length of the whole string instead of the part after the prefix, and putting the cleaned text in the error message instead of the original.

### B2: Runs

**The key idea:** compare each character with the **last run so far**, not with the previous character in the text. That removes the usual fiddly index arithmetic. Because tuples can't be changed, the last run is replaced with a new tuple.

**Common mistakes:** losing the final run (a common bug when a run is only saved when the character changes), and crashing on the empty string. This idea, storing a list of (thing, count) pairs, is called **run-length encoding**, and it's used in image formats and fax machines.

### B3: Predict

- `a`: `range(10, 0, -3)` is 10, 7, 4, 1, so the last is **1**.
- `b`: `"mississippi".count("ss")` is 2 (counts don't overlap), and `"ab" * 3` has 6 characters: **12**.
- `c`: sorting the **keys** by minus their value, so the biggest value first: **["a", "b", "c"]**.
- `d`: the words containing `"o"` are `to`, `or` and `not`: 2 + 2 + 3 = **7**.
- `e`: tuples compare item by item; `"b"` beats `"a"`, and then `2` beats `1`: **("b", 2)**.

If you missed one, trace it in the console with small pieces: `list(range(10, 0, -3))` first, then the indexing.

### B4: Merge

**The key idea:** two positions, `i` and `j`, one in each list. Take the smaller front item and move that position on. When one list runs out, everything left in the other is already sorted and bigger, so add it all at once with slices.

**Common mistakes:** removing items from the input lists with `pop(0)`, which changes the caller's lists (and is slow); forgetting the leftovers; and an `IndexError` from checking `a[i]` after `i` has reached the end. This merge is the heart of **merge sort**, which you'll write in Phase 2.

### B5: Spiral

**The key idea:** take the top row, then **rotate what's left** a quarter turn anticlockwise, so that the next edge to read is again the top row. `zip(*rows)` gives the columns, and reversing their order completes the rotation. The copy at the start (`[list(r) for r in grid]`) protects the caller's grid.

A version with four boundaries (`top`, `bottom`, `left`, `right`) moving inwards is also correct, and closer to what you'd write in many other languages. Its common mistakes are reading the middle row or column twice on non-square grids: that's what the 2 by 4 and 4 by 2 checks were for.

### B6: Attendance

**The key idea:** count **days** and **appearances** separately, then divide. `partition(" ")` splits off the date at the first space only, and gives an empty string for a day with nobody, which the `if name:` check skips.

**Common mistakes:** counting the blank line as a day, counting the empty "name" on 2026-10-08, and forgetting the spaces after commas (giving `" Bo"` and `"Bo"` as two different students).

### B7: Three bugs

The three bugs:

1. `len(word) > 3` should be `>= 3`: "at least 3 letters".
2. The sort key `(kv[1], kv[0])` puts the **least** common first. It needs `(-kv[1], kv[0])`: most common first, ties alphabetical. (`reverse=True` would also reverse the alphabetical order of ties, which is wrong.)
3. `pairs[:n - 1]` returns one too few: `pairs[:n]`.

**The method:** none of these crash, so the evidence is the difference between the expected and actual output. Run the smallest failing check, and compare: a missing three-letter word points to the length test; the wrong order points to the sort; one result too few points to the slice.

### B8: Valid brackets

**The key idea:** a **stack**. Every opening bracket is pushed (with its position); every closing bracket must match the one on top, which is the most recently opened. This is exactly how Python itself checks your brackets, and how your browser checks HTML.

**Common mistakes:** counting brackets instead of using a stack (counting can't tell `([)]` from `([])`), and reporting the **first** unclosed bracket instead of the last one opened.

## Part C

**The design.** Three small functions, each with one job, then a short main program:

- `minutes` validates and converts; it **raises** for a bad time, so `process` can turn that into a problem message.
- `fee` is pure arithmetic: `(stay + 59) // 60` rounds **up** to whole hours without floats, and `min(..., 1500)` applies the cap.
- `process` keeps a dictionary `inside` from plate to entry time: the **index pattern** again. An `in` adds a car, an `out` removes it and records the charge, and whatever is left at the end never left.

**Common mistakes:**

- Not normalising plates, so `abj 45 kd` and `ABJ 45 KD` were treated as different cars.
- Rounding the hours with `round` or `int` (which round to nearest or down) instead of up.
- Charging the free 30 minutes back out of longer stays: the brief says the charge is counted from entry.
- Stopping at the first problem, or crashing on `25:00`.
- Forgetting the cars that never left, or not sorting them.

If you finished Part C, you've written a program that handles real-world messiness. That's the most important skill of Phase 1.

## Second chances

One fresh problem for each skill the exam tested. Do the ones for the problems where you lost marks first.

## Quiz

## What's next

**Your revision plan.** For each problem where you lost marks, write one line: the problem, the kind of mistake (from the list at the top), and the day you'll revisit. Do those days during the rest week. If you're retaking the exam, do it after the rest week, without looking at this page again first.

**Phase 2: Thinking like a programmer.** After a week's rest, Phase 2 turns you from someone who can write Python into someone who can solve problems with it. You'll learn a method for attacking problems you've never seen, recursion, how to measure how fast an algorithm is, the classic searching and sorting algorithms, and the everyday tools of real programming: modules, the command line, Git and automated tests.

You've finished Phase 1. Twelve weeks ago, `print("Hello")` was new. Take the rest week: you've earned it.